← [06 — Planificateur de repas hiérarchique](06_Meal_Planner_Modelisation.ipynb) | [README Z3](README.md)

---

# Notebook 11 — Ordonnancement d'atelier (Job Shop Scheduling) avec Z3.Linq

**Navigation** : [Index Z3](./README.md) | [Index SymbolicAI](../../README.md) | [Index général](../../../README.md)

## Objectifs d'apprentissage

- Modéliser un problème d'**ordonnancement combinatoire** (Job Shop Scheduling) sous forme de CSP
- Encoder des **contraintes de non-chevauchement disjonctives** (`||`) avec Z3.Linq
- Comparer un heuristique glouton avec la recherche SMT exacte
- Implémenter une **recherche linéaire** pour minimiser le makespan Cmax

## Vue d'ensemble : Pipeline de résolution

```mermaid
flowchart LR
    I["Instance JSP\n(N jobs × M machines)"] --> G["Heuristique gloutonne\n(FIFO)"]
    I --> Z["CSP Z3.Linq\n(variables + contraintes)"]
    G --> CG["Cmax glouton\n(sous-optimal)"]
    Z --> BS["Recherche linéaire\nCmax = lb, lb+1, …"]
    BS --> CO["Cmax optimal\n(SMT SAT)"]
    CG --> C["Comparaison"]
    CO --> C
    style CO fill:#c8f7c5
    style CG fill:#f7d5c5
```

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
Console.WriteLine("Imports OK : Z3.Linq (.deploy, OR disjonctif supporté), Microsoft.Z3, System.Linq");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : Z3.Linq (.deploy, OR disjonctif supporté), Microsoft.Z3, System.Linq


## Instance de référence : 3 jobs × 2 machines

Nous utilisons l'instance suivante — chaque job suit un **itinéraire fixe** sur les machines :

| Job | 1ère opération | 2ème opération | Durée totale |
|-----|----------------|----------------|--------------|  
| J0  | Machine A (3h) → | Machine B (2h) | 5h |
| J1  | Machine B (2h) → | Machine A (4h) | 6h |
| J2  | Machine A (2h) → | Machine B (3h) | 5h |

**Charge totale par machine** :
- Machine A : 3 + 4 + 2 = **9h** → borne inférieure sur Cmax
- Machine B : 2 + 2 + 3 = **7h**

> La borne inférieure théorique est **Cmax ≥ 9h**. Peut-on l'atteindre ?

In [2]:
// Instance 3×2 Job Shop
// durA[j] = durée du job j sur Machine A, durB[j] = durée sur Machine B
// startsOnA[j] = true si le job j commence sur A
int[] durA = { 3, 4, 2 };               // J0:3h, J1:4h, J2:2h
int[] durB = { 2, 2, 3 };               // J0:2h, J1:2h, J2:3h
bool[] startsOnA = { true, false, true }; // J0: A→B, J1: B→A, J2: A→B
string[] jobNames = { "J0", "J1", "J2" };

Console.WriteLine("=== Instance Job Shop 3×2 ===");
Console.WriteLine($"{"Job",-6} {"Itinéraire",-24} {"Durée totale"}");
Console.WriteLine(new string('-', 44));
for (int j = 0; j < 3; j++)
{
    string route = startsOnA[j]
        ? $"A({durA[j]}h) → B({durB[j]}h)"
        : $"B({durB[j]}h) → A({durA[j]}h)";
    Console.WriteLine($"{jobNames[j],-6} {route,-24} {durA[j]+durB[j]}h");
}
int lowerBound = Math.Max(durA.Sum(), durB.Sum());
Console.WriteLine($"\nCharge Machine A : {durA.Sum()}h | Machine B : {durB.Sum()}h");
Console.WriteLine($"Borne inférieure Cmax ≥ {lowerBound}h");

=== Instance Job Shop 3×2 ===


Job    Itinéraire               Durée totale


--------------------------------------------


J0     A(3h) → B(2h)            5h


J1     B(2h) → A(4h)            6h


J2     A(2h) → B(3h)            5h



Charge Machine A : 9h | Machine B : 7h


Borne inférieure Cmax ≥ 9h


## Approche gloutonne : ordonnancement FIFO

Une heuristique simple consiste à traiter les jobs dans leur **ordre naturel** (J0 → J1 → J2), en démarrant chaque opération **dès que la machine est disponible**.

**Limitation** : le glouton traite un job après l'autre sans exploiter les chevauchements possibles entre machines. Il ne *regarde pas en avant* pour éviter l'inactivité machine.

In [3]:
// Simulation gloutonne FIFO (ordre J0 → J1 → J2)
int machineAvailA = 0, machineAvailB = 0;
int[] jobFinish = new int[3];

Console.WriteLine("=== Glouton FIFO (J0 → J1 → J2) ===\n");
Console.WriteLine($"{"Job",-5} {"1ère op",-22} {"2ème op",-22} {"Fin"}");
Console.WriteLine(new string('-', 55));

for (int j = 0; j < 3; j++)
{
    int s1, s2;
    if (startsOnA[j])  // J0, J2 : A d'abord puis B
    {
        s1 = machineAvailA;
        machineAvailA = s1 + durA[j];
        s2 = Math.Max(machineAvailA, machineAvailB);
        machineAvailB = s2 + durB[j];
        jobFinish[j] = s2 + durB[j];
        Console.WriteLine($"{jobNames[j],-5} A: t={s1}..{machineAvailA,-14} B: t={s2}..{machineAvailB,-14} {jobFinish[j]}h");
    }
    else               // J1 : B d'abord puis A
    {
        s1 = machineAvailB;
        machineAvailB = s1 + durB[j];
        s2 = Math.Max(machineAvailB, machineAvailA);
        machineAvailA = s2 + durA[j];
        jobFinish[j] = s2 + durA[j];
        Console.WriteLine($"{jobNames[j],-5} B: t={s1}..{machineAvailB,-14} A: t={s2}..{machineAvailA,-14} {jobFinish[j]}h");
    }
}

int cmaxGreedy = jobFinish.Max();
Console.WriteLine($"\nCmax glouton = {cmaxGreedy}h  (borne inf. = {lowerBound}h → écart = {cmaxGreedy - lowerBound}h)");

=== Glouton FIFO (J0 → J1 → J2) ===



Job   1ère op                2ème op                Fin


-------------------------------------------------------


J0    A: t=0..3              B: t=3..5              5h


J1    B: t=5..7              A: t=7..11             11h


J2    A: t=11..13             B: t=13..16             16h



Cmax glouton = 16h  (borne inf. = 9h → écart = 7h)


## Formulation Z3.Linq : variables et contraintes

Le solveur SMT travaille sur des **variables entières** représentant les instants de début :

| Variable | Signification |
|----------|---------------|
| `S00` | Début de J0 sur Machine **A** (durée 3h) |
| `S01` | Début de J0 sur Machine **B** (durée 2h) |
| `S10` | Début de J1 sur Machine **A** (durée 4h) |
| `S11` | Début de J1 sur Machine **B** (durée 2h) |
| `S20` | Début de J2 sur Machine **A** (durée 2h) |
| `S21` | Début de J2 sur Machine **B** (durée 3h) |
| `Cmax` | Makespan (à minimiser) |

**Contraintes disjonctives de non-chevauchement** : pour deux jobs j1 et j2 sur la même machine m :

```
S_{j1,m} ≥ S_{j2,m} + dur_{j2,m}  ∨  S_{j2,m} ≥ S_{j1,m} + dur_{j1,m}
```

En Z3.Linq : `.Where(t => t.S00 >= t.S10 + 4 || t.S10 >= t.S00 + 3)`

In [4]:
// Classe de variables Z3 pour le planning
public class JobShopSchedule
{
    // Instants de début : S{job}{machine} (A=0, B=1)
    public int S00 = 0; // J0 sur Machine A (durée 3h)
    public int S01 = 0; // J0 sur Machine B (durée 2h)
    public int S10 = 0; // J1 sur Machine A (durée 4h)
    public int S11 = 0; // J1 sur Machine B (durée 2h)
    public int S20 = 0; // J2 sur Machine A (durée 2h)
    public int S21 = 0; // J2 sur Machine B (durée 3h)
    public int Cmax = 0;

    public override string ToString()
    {
        var sb = new StringBuilder();
        sb.AppendLine($"  Cmax = {Cmax}h");
        sb.AppendLine($"  J0 : A(t={S00}..{S00+3}) → B(t={S01}..{S01+2})");
        sb.AppendLine($"  J1 : B(t={S11}..{S11+2}) → A(t={S10}..{S10+4})");
        sb.AppendLine($"  J2 : A(t={S20}..{S20+2}) → B(t={S21}..{S21+3})");
        return sb.ToString();
    }
}
Console.WriteLine("Classe JobShopSchedule définie (7 variables Z3 : S00, S01, S10, S11, S20, S21, Cmax).");

Classe JobShopSchedule définie (7 variables Z3 : S00, S01, S10, S11, S20, S21, Cmax).


## Recherche du Cmax optimal

Z3 est un solveur de **satisfaisabilité** (SMT), pas d'optimisation directe. Pour minimiser Cmax, on effectue une **recherche linéaire ascendante** depuis la borne inférieure :

1. Poser `T = borne_inf` (= 9h, charge maximale d'une machine)
2. Demander à Z3 : *Existe-t-il un planning valide avec `Cmax ≤ T` ?*
3. Si **SAT** → T est le makespan optimal → stop
4. Si **UNSAT** → incrémenter T et recommencer

La première valeur T pour laquelle Z3 répond SAT est le makespan optimal.

In [5]:
// Recherche linéaire du Cmax optimal via Z3.Linq
int upperBound = durA.Sum() + durB.Sum(); // borne triviale : 16h (tout séquentiel)
JobShopSchedule optimalSchedule = null;
int optimalCmax = upperBound;

Console.WriteLine($"Recherche de Cmax dans [{lowerBound}, {upperBound}]...");
Console.WriteLine();

for (int T = lowerBound; T <= upperBound; T++)
{
    int bound = T;
    using (var ctx = new Z3Context())
    {
        var theorem = ctx.NewTheorem<JobShopSchedule>()
            // 1. Non-négativité
            .Where(t => t.S00 >= 0 && t.S01 >= 0 && t.S10 >= 0
                       && t.S11 >= 0 && t.S20 >= 0 && t.S21 >= 0 && t.Cmax >= 0)
            // 2. Précédence (routes)
            .Where(t => t.S01 >= t.S00 + 3)  // J0 : A (durée 3) avant B
            .Where(t => t.S10 >= t.S11 + 2)  // J1 : B (durée 2) avant A
            .Where(t => t.S21 >= t.S20 + 2)  // J2 : A (durée 2) avant B
            // 3. Non-chevauchement Machine A (J0:3, J1:4, J2:2)
            .Where(t => t.S00 >= t.S10 + 4 || t.S10 >= t.S00 + 3)
            .Where(t => t.S00 >= t.S20 + 2 || t.S20 >= t.S00 + 3)
            .Where(t => t.S10 >= t.S20 + 2 || t.S20 >= t.S10 + 4)
            // 4. Non-chevauchement Machine B (J0:2, J1:2, J2:3)
            .Where(t => t.S01 >= t.S11 + 2 || t.S11 >= t.S01 + 2)
            .Where(t => t.S01 >= t.S21 + 3 || t.S21 >= t.S01 + 2)
            .Where(t => t.S11 >= t.S21 + 3 || t.S21 >= t.S11 + 2)
            // 5. Borne sur le makespan
            .Where(t => t.Cmax >= t.S00 + 3 && t.Cmax >= t.S01 + 2)
            .Where(t => t.Cmax >= t.S10 + 4 && t.Cmax >= t.S11 + 2)
            .Where(t => t.Cmax >= t.S20 + 2 && t.Cmax >= t.S21 + 3)
            .Where(t => t.Cmax <= bound);

        var solution = theorem.Solve();
        if (solution != null)
        {
            optimalSchedule = solution;
            optimalCmax = solution.Cmax;
            Console.WriteLine($"T={T,2}h → SAT ✓  Cmax optimal trouvé !");
            break;
        }
        else
        {
            Console.WriteLine($"T={T,2}h → UNSAT  (aucun planning possible)");
        }
    }
}

Recherche de Cmax dans [9, 16]...


T= 9h → SAT ✓  Cmax optimal trouvé !


## Résultats : glouton vs Z3

Le diagramme Gantt textuel ci-dessous utilise la convention :
- `0` = J0, `1` = J1, `2` = J2, `.` = machine inactive
- Chaque caractère représente 1 heure

In [6]:
// Comparaison et Gantt textuel
Console.WriteLine("=== Comparaison Glouton vs Z3 ===");
Console.WriteLine($"Cmax glouton : {cmaxGreedy,2}h  (sous-optimal)");
Console.WriteLine($"Cmax Z3      : {optimalCmax,2}h  (optimal)");
Console.WriteLine($"Gain         : {cmaxGreedy - optimalCmax}h ({(cmaxGreedy-optimalCmax)*100/cmaxGreedy}%)");
Console.WriteLine();

if (optimalSchedule != null)
{
    Console.WriteLine("=== Planning optimal (Z3) ===");
    Console.WriteLine(optimalSchedule);

    int ganttLen = optimalCmax;
    char[] gA = Enumerable.Repeat('.', ganttLen).ToArray();
    char[] gB = Enumerable.Repeat('.', ganttLen).ToArray();

    for (int t = optimalSchedule.S00; t < optimalSchedule.S00+3 && t < ganttLen; t++) gA[t]='0';
    for (int t = optimalSchedule.S10; t < optimalSchedule.S10+4 && t < ganttLen; t++) gA[t]='1';
    for (int t = optimalSchedule.S20; t < optimalSchedule.S20+2 && t < ganttLen; t++) gA[t]='2';
    for (int t = optimalSchedule.S01; t < optimalSchedule.S01+2 && t < ganttLen; t++) gB[t]='0';
    for (int t = optimalSchedule.S11; t < optimalSchedule.S11+2 && t < ganttLen; t++) gB[t]='1';
    for (int t = optimalSchedule.S21; t < optimalSchedule.S21+3 && t < ganttLen; t++) gB[t]='2';

    string ticks = string.Concat(Enumerable.Range(0, ganttLen).Select(t => (t % 10).ToString()));
    Console.WriteLine($"  Mach. A: |{new string(gA)}|");
    Console.WriteLine($"  Mach. B: |{new string(gB)}|");
    Console.WriteLine($"  Temps:   |{ticks}|  (h)");
    Console.WriteLine("  Légende: 0=J0 · 1=J1 · 2=J2 · .=inactif");
}

=== Comparaison Glouton vs Z3 ===


Cmax glouton : 16h  (sous-optimal)


Cmax Z3      :  9h  (optimal)


Gain         : 7h (43%)


=== Planning optimal (Z3) ===


  Cmax = 9h
  J0 : A(t=2..5) → B(t=7..9)
  J1 : B(t=0..2) → A(t=5..9)
  J2 : A(t=0..2) → B(t=2..5)



  Mach. A: |220001111|


  Mach. B: |11222..00|


  Temps:   |012345678|  (h)


  Légende: 0=J0 · 1=J1 · 2=J2 · .=inactif


## Exercice 1 — Ajout d'un quatrième job

Étendez le modèle pour inclure un **job J3** avec l'itinéraire :
- Machine B d'abord (durée 1h)
- Machine A ensuite (durée 3h)

**Étapes** :
1. Ajoutez les variables `S30` (début J3 sur A) et `S31` (début J3 sur B) à une classe `JobShopScheduleExt`
2. Ajoutez la contrainte de précédence : `S30 >= S31 + 1`
3. Ajoutez les contraintes de non-chevauchement entre J3 et J0, J1, J2 sur chaque machine
4. Ajoutez la borne makespan pour J3 et lancez la recherche

> **Indice** : la borne inférieure sur Machine A devient 3+4+2+3=**12h**.

In [7]:
// Exercice 1 : ajout du job J3 (B(1h) → A(3h))
// TODO etudiant :
// Etape 1 : définir JobShopScheduleExt avec S30 (A) et S31 (B) supplémentaires
// Etape 2 : contrainte de précédence .Where(t => t.S30 >= t.S31 + 1)
// Etape 3 : no-overlap Machine A pour (J0,J3), (J1,J3), (J2,J3)
// Etape 4 : no-overlap Machine B pour (J0,J3), (J1,J3), (J2,J3)
// Etape 5 : makespan .Where(t => t.Cmax >= t.S30 + 3 && t.Cmax >= t.S31 + 1)
// Etape 6 : recherche linéaire depuis Math.Max(durA.Sum()+3, durB.Sum()+1) = 12

// public class JobShopScheduleExt { /* TODO */ }

Console.WriteLine("Exercice 1 a completer : ajout de J3 (B(1h) → A(3h))");

Exercice 1 a completer : ajout de J3 (B(1h) → A(3h))


## Exercice 2 — Contrainte d'urgence (deadline)

Ajoutez une **contrainte de deadline** : le job J0, considéré urgent, doit **terminer au plus tard à l'heure 7** (les deux opérations doivent être finies avant t=7).

**Étapes** :
1. Reprenez le modèle original (3 jobs × 2 machines)
2. Ajoutez la contrainte `S01 + 2 <= 7` (J0 finit sur Machine B avant t=7)
3. Lancez la recherche et observez l'impact sur Cmax
4. Discutez : quel job est retardé ? La contrainte est-elle toujours satisfaisable ?

> **Indice** : une deadline peut être UNSAT si elle entre en conflit avec les contraintes de précédence ou de non-chevauchement.

In [8]:
// Exercice 2 : deadline sur J0 (doit terminer avant t=7)
// TODO etudiant :
// 1. Reprendre le theorem du notebook avec toutes les contraintes
// 2. Ajouter .Where(t => t.S01 + 2 <= 7)  // J0 finit sur B avant t=7
//    ET    .Where(t => t.S01 >= 0)         // déjà inclus dans non-négativité
// 3. Recherche linéaire depuis lowerBound, afficher le Cmax trouvé
// 4. Commenter : quel job est poussé plus tard ? Cmax change-t-il ?

Console.WriteLine("Exercice 2 a completer : deadline J0 (fin B <= 7h)");

Exercice 2 a completer : deadline J0 (fin B <= 7h)


## Exercice 3 — Extension à 3 machines

Étendez l'instance à **3 jobs × 3 machines** en ajoutant une Machine C :

| Job | 1ère op | 2ème op | 3ème op |
|-----|---------|---------|----------|
| J0  | A (3h) → | B (2h) → | C (1h) |
| J1  | B (2h) → | C (2h) → | A (4h) |
| J2  | C (1h) → | A (2h) → | B (3h) |

**Étapes** :
1. Créez `JobShopSchedule3M` avec `S0C`, `S1C`, `S2C` pour Machine C (en plus de S0A=S00, S0B=S01, etc.)
2. Ajoutez les contraintes de précédence pour la 3ème opération de chaque job
3. Ajoutez les contraintes de non-chevauchement sur Machine C (3 paires)
4. Calculez la borne inférieure et lancez la recherche

> **Borne inférieure** : Machine A : 3+4+2=9h, Machine B : 2+2+3=7h, Machine C : 1+2+1=4h → Cmax ≥ 9h.

In [9]:
// Exercice 3 : extension 3 machines (A, B, C)
// Routes : J0: A(3)→B(2)→C(1), J1: B(2)→C(2)→A(4), J2: C(1)→A(2)→B(3)
// TODO etudiant :
// Etape 1 : JobShopSchedule3M avec S00,S01,S0C, S10,S11,S1C, S20,S21,S2C, Cmax
// Etape 2 : précédence J0: S01>=S00+3, S0C>=S01+2
//           précédence J1: S1C>=S11+2, S10>=S1C+2
//           précédence J2: S20>=S2C+1, S21>=S20+2
// Etape 3 : non-chevauchement Machine C (J0,J1), (J0,J2), (J1,J2)
// Etape 4 : recherche depuis 9h et afficher Gantt 3 machines

// public class JobShopSchedule3M { /* TODO */ }

Console.WriteLine("Exercice 3 a completer : extension 3×3 (Machines A, B, C)");

Exercice 3 a completer : extension 3×3 (Machines A, B, C)


## Conclusion

Le **Job Shop Scheduling** est un problème NP-difficile en général. Sur notre instance 3×2 :

| Approche | Cmax | Méthode |
|----------|------|----------|
| Glouton FIFO | 12h | Décisions locales séquentielles |
| Z3.Linq SMT | 9h | Exploration SMT complète |
| Borne inférieure | 9h | Charge Machine A = 3+4+2 |

**Points clés** :
- Les **contraintes disjonctives** (`||`) expriment naturellement le non-chevauchement en Z3.Linq
- La **recherche linéaire sur Cmax** transforme l'optimisation en séquence de problèmes de satisfaisabilité
- Pour des instances plus grandes, des solveurs spécialisés (Google OR-Tools CP-SAT, CPLEX) sont préférables

**Référence** : *Garey & Johnson (1979)* — Computers and Intractability (NP-complétude du JSP général)